# Lesson 3.1 — Tool/Agent Configuration and Capability Bloat

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Identify capability bloat in an agent's tool configuration
- Apply least-privilege to remove unnecessary tools
- Score a tool set against the roles that actually use it

**You will produce:** A least-privilege tool audit that removes tools no role requires.

## Phase 3 Overview

Domain 3 (19% — the heaviest) is integration: wiring Claude into real systems safely and at scale. This phase covers tool configuration and least privilege, authentication/authorization gap analysis, accuracy-latency trade-offs, observability at scale, RAG pipeline and retrieval design, and connection-protocol selection (MCP vs API/CLI vs agent-to-agent). It is the most decision-dense phase and maps to exam Samples 1 and 3.

By the end of this phase, you will have built:

- A least-privilege tool audit
- An authN/authZ gap-analysis checklist
- An accuracy-vs-latency trade-off model
- An observability log schema and at-scale monitoring plan
- A RAG chunker + retrieval-strategy selector
- A connection-protocol decision matrix (MCP / API-CLI / A2A)

### Prerequisites

- Completed Phases 1-2 (design + prompting)
- Python 3; `pip install anthropic` optional
- Basic security concepts (authentication vs authorization)

## Concepts

An **agent** is a Claude configuration plus a set of **tools** (functions it can call: read a ticket, issue a refund, query a DB). **Capability bloat** is giving an agent more tools — more power — than any legitimate user role needs. Each extra tool enlarges the **attack surface**: a prompt injection or model error can now trigger a dangerous action.

**Least privilege** is the governing principle: grant only the capabilities a role actually requires, and *remove* the rest rather than merely logging or confirming them. This is exactly the exam's Sample 1 — the best answer removes unneeded refund/delete tools rather than adding detective controls around them. Detective controls (logging) and compensating controls (confirmation prompts) are valuable, but they do not reduce *privilege*.

> **Reference:** [Anthropic — Tool use (function calling)](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os, time, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Describe the agent's tools and who needs them

We map each tool to the roles that legitimately use it. Tools no role needs are bloat.

In [ ]:
tools = {
    "read_ticket":   {"risk": "low",      "needed_by": ["support"]},
    "draft_reply":   {"risk": "low",      "needed_by": ["support"]},
    "issue_refund":  {"risk": "high",     "needed_by": ["billing"]},
    "delete_account":{"risk": "critical", "needed_by": ["admin"]},
}
agent_role = "support"  # this agent serves support staff only
for name, meta in tools.items():
    print(f"{name:16} risk={meta['risk']:8} needed_by={meta['needed_by']}")

## Code-Along: Step 2 — Run the least-privilege audit

For a support-only agent, any tool not needed by the support role is capability bloat and should be removed — not logged, not gated. This is the Sample 1 reasoning in code.

In [ ]:
def least_privilege_audit(tools, agent_role):
    keep, remove = [], []
    for name, meta in tools.items():
        (keep if agent_role in meta["needed_by"] else remove).append(name)
    return keep, remove

keep, remove = least_privilege_audit(tools, agent_role)
print("KEEP   :", keep)
print("REMOVE :", remove, "<- capability bloat; delete from config")
print("\nNote: logging or confirmation prompts do NOT reduce privilege; removal does.")

## Exercise

Suppose the agent must occasionally *initiate* (not finalize) a refund for support. Design a safer capability than `issue_refund`: e.g. a `request_refund_approval` tool that creates a pending request for a billing human. Add it to `tools`, re-run the audit, and explain why this respects least privilege better than granting `issue_refund`.

In [ ]:
# TODO: add a scoped 'request_refund_approval' tool needed_by ['support']
# TODO: re-run least_privilege_audit and print the result
# TODO: one-line rationale on least privilege vs convenience

## Key Takeaways

- Capability bloat = tools beyond what any serving role needs; it enlarges the attack surface.
- Least privilege means removing unneeded capabilities, not monitoring them (Sample 1 = B).
- Logging is detective; confirmation is compensating; neither reduces privilege.
- Prefer scoped, approval-gated tools over broad high-risk ones.

---

**Next:** [Lesson 3.2 — Authentication and Authorization Gap Analysis](./lesson-3.2-authn-authz-gap-analysis.ipynb)